# USE CASE 3: Early Warning Systems and Forecasting

## For: Meteorological Services, Disaster Management Authorities, Civil Protection

### Key Questions This Analysis Answers:
1. **What are the historical patterns that can inform forecasting?**
2. **When are the high-risk seasons for different hazards?**
3. **What impact thresholds should trigger alerts?**
4. **Which areas are most vulnerable and need priority warnings?**
5. **How can we improve lead time for warnings?**

### Why Historical Data Matters:
- **Pattern Recognition** → Identify recurring seasonal trends
- **Threshold Setting** → Define when to issue alerts based on past impacts
- **Vulnerability Mapping** → Target warnings to high-risk populations
- **Forecast Validation** → Compare predictions against historical events

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from config import PROCESSED_DATA_DIR, FIGURES_DIR, REPORTS_DIR

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("✓ Libraries loaded")

## 1. Load Historical Disaster Data

In [ ]:
data_file = PROCESSED_DATA_DIR / 'disaster_events_sendai.csv'

if data_file.exists():
    df = pd.read_csv(data_file)
    print(f"Loaded {len(df)} disaster events")
    print(f"Time period: {df['year'].min()} - {df['year'].max()}")
    print(f"Years of data: {df['year'].max() - df['year'].min() + 1}")
else:
    print("Please run notebook 05_risk_modeling.ipynb first")
    df = None

## 2. Seasonal Risk Windows - When to Issue Warnings

### Early Warning Message:
"Historical patterns show when hazards are most likely to occur."

In [ ]:
if df is not None and 'month' in df.columns and 'hazardtype' in df.columns:
    # Get top hazards
    top_hazards = df['hazardtype'].value_counts().head(5).index
    
    # Create seasonal pattern matrix
    seasonal_matrix = df[df['hazardtype'].isin(top_hazards)].groupby(['hazardtype', 'month']).size().unstack(fill_value=0)
    
    print("="*80)
    print("SEASONAL RISK WINDOWS - EARLY WARNING CALENDAR")
    print("="*80)
    print("\nEvent frequency by month (top 5 hazards):\n")
    
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    seasonal_matrix.columns = [month_names[int(m)-1] if m <= 12 else str(m) for m in seasonal_matrix.columns]
    
    display(seasonal_matrix)
    
    # Heatmap
    fig, ax = plt.subplots(figsize=(14, 8))
    sns.heatmap(seasonal_matrix, annot=True, fmt='.0f', cmap='YlOrRd', 
                cbar_kws={'label': 'Number of Events'}, ax=ax, linewidths=0.5)
    ax.set_title('Seasonal Risk Windows: Early Warning Calendar', fontsize=14, fontweight='bold')
    ax.set_xlabel('Month', fontsize=12)
    ax.set_ylabel('Hazard Type', fontsize=12)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'UC3_seasonal_risk_windows.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Identify high-risk periods
    print("\n🚨 HIGH-RISK PERIODS FOR EARLY WARNING ACTIVATION:\n")
    for hazard in seasonal_matrix.index:
        peak_months = seasonal_matrix.loc[hazard].nlargest(3)
        if peak_months.sum() > 0:
            print(f"{hazard}:")
            print(f"  Peak risk months: {', '.join(peak_months.index)}")
            print(f"  Activate warnings: 2 weeks before {peak_months.index[0]}")
            print(f"  Maintain heightened alert through: {peak_months.index[-1]}\n")

## 3. Impact Thresholds for Alert Triggers

### Setting Warning Levels Based on Historical Impacts

In [ ]:
if df is not None and 'total_affected' in df.columns:
    # Calculate impact percentiles for different hazards
    print("\n" + "="*80)
    print("IMPACT THRESHOLDS FOR EARLY WARNING ALERT LEVELS")
    print("="*80)
    
    top_hazards = df['hazardtype'].value_counts().head(5).index
    
    threshold_data = []
    
    for hazard in top_hazards:
        hazard_df = df[df['hazardtype'] == hazard]
        
        if len(hazard_df) > 0 and hazard_df['total_affected'].notna().sum() > 0:
            affected_data = hazard_df['total_affected'].dropna()
            
            # Define alert levels based on percentiles
            advisory = affected_data.quantile(0.50)  # 50th percentile
            watch = affected_data.quantile(0.75)     # 75th percentile
            warning = affected_data.quantile(0.90)   # 90th percentile
            emergency = affected_data.quantile(0.95) # 95th percentile
            
            print(f"\n{hazard}:")
            print(f"  🟢 ADVISORY (50th percentile):   {advisory:,.0f} people affected")
            print(f"  🟡 WATCH (75th percentile):      {watch:,.0f} people affected")
            print(f"  🟠 WARNING (90th percentile):    {warning:,.0f} people affected")
            print(f"  🔴 EMERGENCY (95th percentile):  {emergency:,.0f} people affected")
            
            threshold_data.append({
                'Hazard': hazard,
                'Advisory_Threshold': advisory,
                'Watch_Threshold': watch,
                'Warning_Threshold': warning,
                'Emergency_Threshold': emergency
            })
    
    threshold_df = pd.DataFrame(threshold_data)
    threshold_df.to_csv(REPORTS_DIR / 'UC3_alert_thresholds.csv', index=False)
    print("\n✓ Alert thresholds saved to outputs/reports/")
    
    # Visualize thresholds
    fig = go.Figure()
    
    for level, color in [('Advisory_Threshold', 'green'), 
                         ('Watch_Threshold', 'yellow'),
                         ('Warning_Threshold', 'orange'),
                         ('Emergency_Threshold', 'red')]:
        fig.add_trace(go.Bar(
            name=level.replace('_Threshold', ''),
            x=threshold_df['Hazard'],
            y=threshold_df[level],
            marker_color=color
        ))
    
    fig.update_layout(
        title='Early Warning Alert Thresholds by Hazard Type',
        xaxis_title='Hazard Type',
        yaxis_title='People Affected',
        barmode='group',
        height=500
    )
    
    fig.show()
    fig.write_html(FIGURES_DIR / 'UC3_alert_thresholds.html')

## 4. Vulnerable Area Mapping - Priority Warning Zones

In [ ]:
if df is not None and 'level2' in df.columns:
    # Identify vulnerable areas based on frequency and impact
    vulnerability_analysis = df.groupby('level2').agg({
        'serial': 'count',
        'total_affected': 'sum',
        'total_deaths': 'sum',
        'hazardtype': 'nunique'
    }).rename(columns={
        'serial': 'event_count',
        'total_affected': 'total_affected',
        'total_deaths': 'total_deaths',
        'hazardtype': 'hazard_diversity'
    })
    
    # Calculate vulnerability score
    for col in ['event_count', 'total_affected', 'total_deaths', 'hazard_diversity']:
        if vulnerability_analysis[col].max() > 0:
            vulnerability_analysis[f'{col}_norm'] = (
                vulnerability_analysis[col] / vulnerability_analysis[col].max()
            )
        else:
            vulnerability_analysis[f'{col}_norm'] = 0
    
    vulnerability_analysis['vulnerability_score'] = (
        vulnerability_analysis['event_count_norm'] * 0.3 +
        vulnerability_analysis['total_affected_norm'] * 0.4 +
        vulnerability_analysis['total_deaths_norm'] * 0.2 +
        vulnerability_analysis['hazard_diversity_norm'] * 0.1
    )
    
    vulnerability_analysis = vulnerability_analysis.sort_values('vulnerability_score', ascending=False)
    
    print("\n" + "="*80)
    print("VULNERABLE AREAS - PRIORITY WARNING ZONES")
    print("="*80)
    print("\nTop 20 areas requiring priority early warning systems:\n")
    
    display(vulnerability_analysis.head(20))
    
    print("\n🎯 EARLY WARNING SYSTEM PRIORITIES:")
    print("\n   TIER 1 - CRITICAL (Top 5):")
    for i, location in enumerate(vulnerability_analysis.head(5).index, 1):
        score = vulnerability_analysis.loc[location, 'vulnerability_score']
        events = vulnerability_analysis.loc[location, 'event_count']
        print(f"   {i}. {location} (Score: {score:.2f}, {events} events)")
        print(f"      → Install automated alert systems")
        print(f"      → 24/7 monitoring during high-risk seasons")
    
    print("\n   TIER 2 - HIGH PRIORITY (Next 10):")
    for location in vulnerability_analysis.iloc[5:15].index:
        print(f"   → {location}")
    print("      → Community-based early warning")
    print("      → Mobile alert systems")
    
    vulnerability_analysis.to_csv(REPORTS_DIR / 'UC3_vulnerable_areas.csv')
    print("\n✓ Vulnerable area analysis saved")

## 5. Historical Patterns for Forecasting Models

### Trend Analysis and Pattern Recognition

In [ ]:
if df is not None and 'year' in df.columns:
    # Annual frequency trends
    annual_freq = df.groupby(['year', 'hazardtype']).size().reset_index(name='count')
    
    top_hazards = df['hazardtype'].value_counts().head(3).index
    
    print("\n" + "="*80)
    print("HISTORICAL PATTERNS FOR FORECASTING")
    print("="*80)
    
    fig = make_subplots(
        rows=len(top_hazards), cols=1,
        subplot_titles=[f'{hazard} - Trend Analysis' for hazard in top_hazards],
        vertical_spacing=0.1
    )
    
    for i, hazard in enumerate(top_hazards, 1):
        hazard_data = annual_freq[annual_freq['hazardtype'] == hazard]
        
        if len(hazard_data) > 2:
            # Calculate trend
            x = hazard_data['year'].values
            y = hazard_data['count'].values
            
            if len(x) > 0:
                slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
                trend_line = slope * x + intercept
                
                # Add actual data
                fig.add_trace(
                    go.Scatter(x=hazard_data['year'], y=hazard_data['count'],
                              mode='lines+markers', name=f'{hazard} (Actual)',
                              line=dict(color='steelblue')),
                    row=i, col=1
                )
                
                # Add trend line
                fig.add_trace(
                    go.Scatter(x=hazard_data['year'], y=trend_line,
                              mode='lines', name=f'{hazard} (Trend)',
                              line=dict(color='red', dash='dash')),
                    row=i, col=1
                )
                
                print(f"\n{hazard}:")
                if slope > 0:
                    print(f"  📈 INCREASING trend: +{slope:.2f} events/year")
                    print(f"     → Enhanced monitoring recommended")
                elif slope < 0:
                    print(f"  📉 DECREASING trend: {slope:.2f} events/year")
                    print(f"     → DRR measures may be working")
                else:
                    print(f"  ➡️  STABLE trend")
                
                print(f"  R² = {r_value**2:.3f} (trend strength)")
    
    fig.update_layout(height=300*len(top_hazards), showlegend=False,
                     title_text='Historical Patterns for Forecasting Models')
    fig.show()
    fig.write_html(FIGURES_DIR / 'UC3_forecasting_patterns.html')

## 6. Lead Time Analysis - How Much Warning is Possible?

### Based on Event Duration and Onset Patterns

In [ ]:
if df is not None and 'durationdays' in df.columns:
    duration_analysis = df.groupby('hazardtype')['durationdays'].agg(['mean', 'median', 'max']).sort_values('mean', ascending=False)
    
    print("\n" + "="*80)
    print("EVENT DURATION ANALYSIS - POTENTIAL WARNING LEAD TIME")
    print("="*80)
    print("\nAverage event duration by hazard type:\n")
    
    display(duration_analysis.head(10))
    
    print("\n⏰ RECOMMENDED WARNING LEAD TIMES:\n")
    
    for hazard in duration_analysis.head(5).index:
        avg_duration = duration_analysis.loc[hazard, 'mean']
        
        if avg_duration >= 30:
            lead_time = "7-14 days"
            warning_type = "LONG-RANGE FORECAST"
        elif avg_duration >= 7:
            lead_time = "3-7 days"
            warning_type = "MEDIUM-RANGE FORECAST"
        elif avg_duration >= 2:
            lead_time = "24-72 hours"
            warning_type = "SHORT-RANGE WARNING"
        else:
            lead_time = "6-24 hours"
            warning_type = "IMMEDIATE ALERT"
        
        print(f"{hazard}:")
        print(f"  Average duration: {avg_duration:.1f} days")
        print(f"  Warning type: {warning_type}")
        print(f"  Recommended lead time: {lead_time}\n")

## 7. Multi-Hazard Early Warning Integration

In [ ]:
if df is not None and 'level2' in df.columns:
    # Identify areas with multiple hazard types
    multi_hazard_areas = df.groupby('level2')['hazardtype'].nunique().sort_values(ascending=False).head(15)
    
    print("\n" + "="*80)
    print("MULTI-HAZARD EARLY WARNING REQUIREMENTS")
    print("="*80)
    print("\nAreas requiring integrated early warning systems (multiple hazard types):\n")
    
    for location, hazard_count in multi_hazard_areas.items():
        hazards = df[df['level2'] == location]['hazardtype'].unique()
        print(f"{location}: {hazard_count} hazard types")
        print(f"  Hazards: {', '.join(hazards[:5])}")
        print(f"  → Requires MULTI-HAZARD early warning system\n")

## 8. Executive Summary for Early Warning Authorities

In [ ]:
if df is not None:
    print("\n" + "="*80)
    print("EXECUTIVE BRIEFING: EARLY WARNING SYSTEM STRATEGY")
    print("For: National Meteorological Service / Disaster Management Authority")
    print("="*80)
    
    print("\n1. SEASONAL RISK WINDOWS:")
    print("   Historical data reveals clear seasonal patterns for each hazard type.")
    print("   → Activate early warning systems 2 weeks before peak risk months")
    
    print("\n2. ALERT THRESHOLDS:")
    print("   Impact thresholds defined based on historical percentiles:")
    print("   → Advisory (50th), Watch (75th), Warning (90th), Emergency (95th)")
    
    print("\n3. PRIORITY WARNING ZONES:")
    if 'level2' in df.columns:
        top_5 = vulnerability_analysis.head(5).index.tolist()
        print(f"   Top 5 vulnerable areas: {', '.join(top_5[:3])}...")
    print("   → Install automated alert systems in Tier 1 zones")
    
    print("\n4. FORECASTING PATTERNS:")
    print("   Trend analysis shows increasing/decreasing patterns for each hazard.")
    print("   → Use historical patterns to validate forecast models")
    
    print("\n5. LEAD TIME RECOMMENDATIONS:")
    print("   Event duration analysis suggests optimal warning lead times:")
    print("   → Slow-onset: 7-14 days | Rapid-onset: 6-24 hours")
    
    print("\n6. DATA ADVANTAGE:")
    print("   ✓ Evidence-based alert thresholds")
    print("   ✓ Seasonal risk calendars for planning")
    print("   ✓ Vulnerable area mapping for targeted warnings")
    print("   ✓ Historical validation for forecast models")
    
    print("\n" + "="*80)